<!-- nav -->
[← 8.02 · Python Patterns](08_02_Python_Patterns.ipynb) · [Home](../../README.md) · [8.04 · TypeScript Types →](08_04_TypeScript_Types.ipynb)

# 8.03 · Python RegExp

Python's regular expressions are strings its `re` module reads, in a grammar of their own, the Library Reference's
"Regular Expression Syntax":

```text
re.fullmatch(r"(?P<word>[a-z]+)-(?P=word)", text)    a word, a hyphen, the same word again
```

`Python RegExp` is that grammar's dialect: its kinds are the constructs `re`'s syntax lists, named and shaped as
regexpp, ECMAScript's reference parser, names the ones ECMAScript has too (a `pattern` of `alternative`s, a
`capturinggroup`, a `quantifier`, a `characterclass`), then Python's own, its `conditional`, `atomicgroup`, inline
`flags` and `comment`
([CAP-281: Python's, ECMAScript's and C++'s regular expressions as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-281.md)).
Its checks are `re`'s parser's, with its messages.

In Python3, `r"(?P<word>[a-z]+)"` is a string, and stays one: no reader guesses that a string is a regular expression
([PRI-025: never guess the schema](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-025)). A
regular expression is written as one in its dialect's holder, `S.Python.RegExp.Expression`, by whoever knows it is one.
This page builds regular expressions from a single character up, in the builders' fluent calls, prints each as `re`
reads it, validates them, and reads the editions that have its kinds. Like the rest of Part I, it starts from
`dialects()`; `regexp` holds a spec in the dialect's holder, and `char` writes a character. Python's notebook also compiles
each text with `re` itself, which only Python can, and checks that its parser accepts what the dialect accepts and
refuses what it refuses.

In [1]:
import assert from "node:assert/strict";
import { Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as PythonRegExp from "@mbse/expressions/Python/RegExp";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const REGEXP = PythonRegExp.Expressions.DIALECT;
const ToText = PythonRegExp.Text.ToText;
const char = (c: string) => (u: any) => u.character((h: any) => h.value(BigInt(c.codePointAt(0)!))); // the character c, its value a code point

/** The regular expression `spec` writes, held by its dialect's holder, written in a transaction of its own. */
function regexp(spec: any): any {
  return store.transaction(() => S.Python.RegExp.Expression((e: any) => e.root(spec)));
}

/** A regular expression's text, as `re` reads it, and its kind (Python's notebook also compiles it with `re`). */
function show(held: any): void {
  console.log(ToText(held).padEnd(28), Terms.kindOf(Terms.term(held))[0]);
}

<a id="T08.03.1"></a>
## 08.03.1 · A character, a sequence, alternatives

The smallest regular expression is one character, `a`: a `character`, its `value` a code point, 97. Characters in a
row, `ab`, are an `alternative`, its `elements` in order; alternatives joined by `|`, `ab|c`, are a `pattern`, the
root `re` parses a whole regular expression to:

In [2]:
const sequence = (u: any) => u.alternative((a: any) => a.elements([char("a"), char("b")])); // ab
for (const spec of [char("a"), sequence,
  (u: any) => u.pattern((p: any) => p.alternatives([sequence, (a: any) => a.alternative((s: any) => s.elements([char("c")]))]))]) {
  show(regexp(spec));
}
const form = Terms.form(Terms.term(regexp(char("a"))));
console.log(form.kind, `value: ${form.attributes.get("value")}`);

a                            character


ab                           alternative


ab|c                         pattern


character value: 97


<a id="T08.03.2"></a>
## 08.03.2 · Classes, sets and quantifiers

A **character class**, `[a-z_]`, matches one character among its elements: characters, and ranges, a
`characterclassrange` of two characters; `[^...]` negates it. A **character set** is a class `re` names: `.` any
character, `\d` a digit, `\s` a space, `\w` a word character, each a `characterset` of its `setKind`, `\D` and the
others negated. A **quantifier** repeats its `element` between its `min` and its `max`, none for no bound: `*`, `+`, `?`
and `{2,5}` are its shortest spellings, the printer's, then `?` where it is lazy and `+` where it is possessive (Python
3.11):

In [3]:
const lowercase = (e: any) => e.characterclassrange((r: any) => r.min(char("a")).max(char("z")));
const specs = [
  (u: any) => u.characterclass((c: any) => c.elements([lowercase, char("_")])),
  (u: any) => u.characterclass((c: any) => c.negate(true).elements([lowercase])),
  (u: any) => u.characterset((s: any) => s.setKind("any")),
  (u: any) => u.characterset((s: any) => s.setKind("digit")),
  (u: any) => u.characterset((s: any) => s.setKind("word").negate(true)),
  (u: any) => u.quantifier((q: any) => q.min(0n).element(char("a"))),
  (u: any) => u.quantifier((q: any) => q.min(1n).element((e: any) => e.characterset((s: any) => s.setKind("digit")))),
  (u: any) => u.quantifier((q: any) => q.min(0n).max(1n).element(char("a"))),
  (u: any) => u.quantifier((q: any) => q.min(2n).max(5n).greedy(false).element(char("a"))),
  (u: any) => u.quantifier((q: any) => q.min(1n).possessive(true).element(char("a"))),
];
for (const spec of specs) show(regexp(spec));

[a-z_]                       characterclass


[^a-z]                       characterclass


.                            characterset


\d                           characterset


\W                           characterset


a*                           quantifier


\d+                          quantifier


a?                           quantifier


a{2,5}?                      quantifier


a++                          quantifier


<a id="T08.03.3"></a>
## 08.03.3 · Groups, assertions and references

A **group** gathers alternatives: `(?:a|b)` a `group`, `(a)` a `capturinggroup`, numbered as it opens, and
`(?P<word>a)` one with a name. An **assertion** matches a place, not a character: `^`, `$`, `\b`, each an `assertion`
of its `assertionKind`, and the lookarounds `(?=a)` and `(?<!a)`, which hold alternatives. A **backreference** matches
what a group matched, `\1` by its number or `(?P=word)` by its name, both its `ref`. Python adds a `conditional`,
`(?(1)a|b)`, an `atomicgroup`, `(?>a+)`, inline `flags`, `(?i)` and `(?i-m:a)`, and a `comment`, `(?#note)`. The opening
example, whole:

In [4]:
const word = (u: any) => u.capturinggroup((g: any) => g.name("word").alternatives([(a: any) => a.alternative((s: any) => s.elements([
  (e: any) => e.quantifier((q: any) => q.min(1n).element((c: any) => c.characterclass((k: any) => k.elements([lowercase]))))]))]));
const repeated = regexp((u: any) => u.pattern((p: any) => p.alternatives([(a: any) => a.alternative((s: any) => s.elements([
  word, char("-"), (e: any) => e.backreference((b: any) => b.ref("word"))]))])));
show(repeated);
const alternative = (...elements: any[]) => (a: any) => a.alternative((s: any) => s.elements(elements));
const more = [
  (u: any) => u.group((g: any) => g.alternatives([alternative(char("a")), alternative(char("b"))])),
  (u: any) => u.alternative((s: any) => s.elements([
    (e: any) => e.capturinggroup((g: any) => g.alternatives([alternative(char("a"))])),
    (e: any) => e.backreference((b: any) => b.ref(1n))])),
  (u: any) => u.alternative((s: any) => s.elements([
    (e: any) => e.assertion((a: any) => a.assertionKind("start")), char("a"),
    (e: any) => e.assertion((a: any) => a.assertionKind("word")),
    (e: any) => e.assertion((a: any) => a.assertionKind("lookahead").alternatives([alternative(char("b"))])),
    (e: any) => e.assertion((a: any) => a.assertionKind("lookbehind").negate(true).alternatives([alternative(char("c"))])),
    (e: any) => e.assertion((a: any) => a.assertionKind("end"))])),
  (u: any) => u.alternative((s: any) => s.elements([
    (e: any) => e.quantifier((q: any) => q.min(0n).max(1n).element(
      (c: any) => c.capturinggroup((g: any) => g.alternatives([alternative(char("a"))])))),
    (e: any) => e.conditional((c: any) => c.ref(1n).yes(alternative(char("b"))).no(alternative(char("c"))))])),
  (u: any) => u.atomicgroup((g: any) => g.alternatives([alternative((e: any) => e.quantifier((q: any) => q.min(1n).element(char("a"))))])),
  (u: any) => u.alternative((s: any) => s.elements([(e: any) => e.flags((f: any) => f.add("i")), char("a"),
    (e: any) => e.comment((c: any) => c.text("note"))])),
  (u: any) => u.flags((f: any) => f.add("i").remove("m").alternatives([alternative(char("a"))])),
];
for (const spec of more) show(regexp(spec));

(?P<word>[a-z]+)-(?P=word)   pattern


(?:a|b)                      group


(a)\1                        alternative


^a\b(?=b)(?<!c)$             alternative


(a)?(?(1)b|c)                alternative


(?>a+)                       atomicgroup


(?i)a(?#note)                alternative


(?i-m:a)                     flags


<a id="T08.03.4"></a>
## 08.03.4 · A character's spelling, kept beside its value

A character has many spellings in `re`: `A`, `\x41`, `\101`, `\N{LATIN CAPITAL LETTER A}`. Its value is one code
point, 65; its `raw`, where a reader kept it, is its spelling as written, which the printer writes back, beside the
value it encodes ([PRI-023: domain representations](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-023)).
Without a `raw`, the printer writes its default spelling, escaping what is special where it stands: `.` outside a
class, `-` within one. Validation checks that a `raw` spells its value, as `re`'s parser reads it:

In [5]:
const spelled = ["A", "\\x41", "\\101", "\\N{LATIN CAPITAL LETTER A}"].map(
  (raw) => (u: any) => u.character((c: any) => c.raw(raw).value(65n)));
for (const spec of spelled) {
  const held = regexp(spec);
  console.log(ToText(held).padEnd(28), `value: ${Terms.form(Terms.term(held)).attributes.get("value")}`, "|",
    REGEXP.validate(held).join("; ") || "valid");
}
show(regexp((u: any) => u.alternative((s: any) => s.elements([char("."), (e: any) => e.characterclass((c: any) => c.elements([char(".")]))]))));
show(regexp((u: any) => u.characterclass((c: any) => c.elements([char("-"), char("a")]))));
const wrong = regexp((u: any) => u.character((c: any) => c.raw("b").value(97n)));
console.log(ToText(wrong), "|", ...REGEXP.validate(wrong));
assert.deepEqual(REGEXP.validate(wrong), ["a character's raw 'b' does not spell U+0061"]);

A                            value: 65 | valid


\x41                         value: 65 | valid


\101                         value: 65 | valid


\N{LATIN CAPITAL LETTER A}   value: 65 | valid


\.[.]                        alternative


[\-a]                        characterclass


b | a character's raw 'b' does not spell U+0061


<a id="T08.03.5"></a>
## 08.03.5 · Validated, as `re` validates it

Validation reports what `re`'s parser refuses, in its words: a quantifier's bounds out of order, a quantifier with
nothing to repeat, a group's name defined twice, a reference to a group the pattern does not have. A pattern's groups
and references are checked at its root, where they are numbered. Each refused term is kept, and printed as written
([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [6]:
const refused = [
  (u: any) => u.quantifier((q: any) => q.min(5n).max(2n).element(char("a"))),
  (u: any) => u.quantifier((q: any) => q.min(0n).element((a: any) => a.assertion((s: any) => s.assertionKind("start")))),
  (u: any) => u.pattern((p: any) => p.alternatives([alternative(
    (e: any) => e.capturinggroup((g: any) => g.name("n").alternatives([alternative(char("a"))])),
    (e: any) => e.capturinggroup((g: any) => g.name("n").alternatives([alternative(char("b"))])))])),
  (u: any) => u.pattern((p: any) => p.alternatives([alternative(
    (e: any) => e.capturinggroup((g: any) => g.alternatives([alternative(char("a"))])), (e: any) => e.backreference((b: any) => b.ref(2n)))])),
];
for (const spec of refused) {
  const held = regexp(spec); // Python's notebook also checks that `re` refuses it, with the same message
  console.log(ToText(held).padEnd(16), ...REGEXP.validate(held));
}

a{5,2}           min repeat greater than max repeat


^*               nothing to repeat


(?P<n>a)(?P<n>b) redefinition of group name 'n' as group 2; was group 1


(a)\2            invalid group reference 2


<a id="T08.03.6"></a>
## 08.03.6 · Held by its holder alone, binding nothing, not simplified

No Python3 expression holds a regular expression: Python3 has strings, and `re.compile(r"a+")` calls a function with
one, a Python3 `call` of a `literal` (tutorial 6), which nothing reads as anything but a string. So only its holder holds
one, while each dialect of the family reads the others' kinds and prints through one printer (tutorial 8, 08.5). A
named group's name is a label its backreferences resolve, not a name the expression binds: a regular expression needs
nothing from a scope. And it is matched by `re`, a library, not simplified: Python3's rules fold none of its kinds: each stays in the residual, named by its dialect:

In [7]:
const variable = (name: string) => (d: any) => d.variable((v: any) => v.name(name));
const call = store.transaction(() => S.Python.Python3.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
  .func((f: any) => f.attribute((a: any) => a.value(variable("re")).attr("compile")))
  .args([(a: any) => a.literal((r: any) => r.value((v: any) => v.as_scalar("a+")).text('r"a+"').quote('"'))])))));
const argument = Terms.form(Terms.term(call)).arguments.at(-1);
console.log(Python3.Text.ToText(call), "| its argument:", Terms.kindOf(argument)[0], "of", Terms.dialectOf(argument));
console.log("Python RegExp reads", REGEXP.reads().map((d: any) => d.name()).join(", "), "| one printer:",
  ToText === Python3.Text.ToText ? "True" : "False");
console.log(ToText(repeated), "| free:", [...Symbolics.free(repeated, REGEXP)].join(", ") || "none");
try {
  Python3.Simplifiers.OfAny(repeated).evaluate();
  throw new Error("evaluated");
} catch (error) {
  assert.equal((error as Error).name, "NotEvaluated");
  console.log((error as Error).message);
}

re.compile(r"a+") | its argument: literal of Expressions.Python.Python3.Term


Python RegExp reads Python3, Python Types, Python Patterns | one printer: True


(?P<word>[a-z]+)-(?P=word) | free: none


Python RegExp pattern is not evaluated


<a id="T08.03.7"></a>
## 08.03.7 · Its editions as data

`Python RegExp` covers Python 3.0 to 3.15, each found by its published name, and records which of its kinds each has
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
every one since 3.0 but `atomicgroup`, from 3.11. What an edition added within a kind, such as 3.11's possessive
quantifiers, is not recorded apart:

In [8]:
const dialect = store.Instances.Expressions.Python.RegExp;
const editions = [...Standards.standards(dialect).keys()];
console.log(editions.length, "editions:", editions[0], "to", editions.at(-1));
for (const version of ["Python 3.10", "Python 3.11"]) {
  const kinds = Standards.kinds(Standards.find(dialect, version)!).map((kind: any) => kind.name.split(".").at(-1));
  console.log(version.padEnd(11), kinds.length, "kinds, AtomicGroup among them:", kinds.includes("AtomicGroup") ? "True" : "False");
}
const atomic = store.registered("Expressions.Python.RegExp.AtomicGroup");
console.log("AtomicGroup since", [...Standards.of(dialect, atomic).keys()][0]);

16 editions: Python 3.0 to Python 3.15


Python 3.10 14 kinds, AtomicGroup among them: False


Python 3.11 15 kinds, AtomicGroup among them: True


AtomicGroup since Python 3.11


## What you learned

- **A regular expression is a term of its own dialect**, `Python RegExp`, named as regexpp names what ECMAScript shares
  with it, Python's own constructs beside them; held by its holder, never guessed from a string.
- **It is built from a character up**: alternatives of elements, classes and sets, quantifiers, groups, assertions and
  references, and Python's conditionals, atomic groups, flags and comments.
- **A character keeps its spelling** beside its code point, and validation checks one spells the other.
- **Validation is `re`'s**, with its messages; a named group's name is a label, so it binds nothing.
- **It is not simplified**: `re` matches it; Python3's rules fold none of its kinds: each stays in the residual, named by its dialect.
- **Its editions are data**: Python 3.0 to 3.15, `atomicgroup` from 3.11.

Next, the TypeScript family, starting with `TypeScript Types`.

<!-- nav -->
[← 8.02 · Python Patterns](08_02_Python_Patterns.ipynb) · [Home](../../README.md) · [8.04 · TypeScript Types →](08_04_TypeScript_Types.ipynb)